# WikiGPT 03: Chat tuning and export

Teaches the pretrained model to reply in turns, then exports `wikigpt-12m.zip` for the local app.

**Settings:** Accelerator GPU T4 x2, Internet On.  
**Inputs:** the `wikigpt-code` dataset and the output of notebook 02.  
**Run with:** Save Version, then Save & Run All (Commit).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")


def find_input(pattern: str) -> Path:
    matches = sorted(INPUT.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}. Attach the inputs listed at the top of this notebook.")
    return matches[0]


def run(*args) -> None:
    command = [sys.executable, "-u", *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"command failed with exit code {process.returncode}")


CODE = find_input("wikigpt/__init__.py").parents[1]
sys.path.insert(0, str(CODE))
sys.dont_write_bytecode = True
os.environ.update(PYTHONPATH=str(CODE), PYTHONDONTWRITEBYTECODE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
PRETRAIN = find_input("pretrain/ckpt.pt")

print("code:", CODE)
print("pretrain checkpoint:", PRETRAIN)
print("inputs:", ", ".join(sorted(p.name for p in INPUT.iterdir())))
print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(f"free disk: {shutil.disk_usage(WORKING).free / 1e9:.1f} GB")

## Tests

In [ ]:
run("-m", "pytest", CODE / "tests", "-q", "-p", "no:cacheprovider")

## Chat tuning

Downloads the two chat datasets and trains for 2 epochs. Expect well under an hour on a T4.

In [ ]:
FINETUNE = WORKING / "runs" / "finetune"
run(CODE / "scripts" / "finetune.py", "--checkpoint", PRETRAIN, "--out", FINETUNE)

## Sample replies

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(FINETUNE / "loss.png")))
print((FINETUNE / "replies.txt").read_text(encoding="utf-8"))

## Export

Writes the bundle the local app loads and checks that it reloads to the same model.

In [ ]:
EXPORT = WORKING / "export"
run(CODE / "scripts" / "export.py", "--checkpoint", FINETUNE / "ckpt.pt", "--out", EXPORT)

In [ ]:
from IPython.display import FileLink

display(FileLink("export/wikigpt-12m.zip"))